# Multi Agent System Research Project


---

## Project Overview


> Multi LLM agent system that capable of deep research on complex topics by coordinating agents with specialized persona. \
> Demonstrate clear architectural decisions, measurable improvements over a baseline, and robust handling of complex research queries.

---

## Step 1: Import Required Libraries

In [ ]:
# Common library imports

# Load environment variables and verify API keys
import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

if GOOGLE_API_KEY and TAVILY_API_KEY:
    print(f"GOOGLE_API_KEY loaded: {GOOGLE_API_KEY[:5]}...")
    print(f"TAVILY_API_KEY loaded: {TAVILY_API_KEY[:5]}...")
else:
    print("Failed to load API keys. Please verify .env configuration.")

In [ ]:
# Essential Library Import
import re
import operator
from typing import Dict, List, Annotated
from typing_extensions import TypedDict

from pydantic import BaseModel, Field
from langchain_tavily import TavilySearch
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import scipy.interpolate as interpolate

print("Environment initialized successfully.")

---

## Step 2. Tools and Agent Structure Implementation

In [ ]:
# Define state schema, pydantic output structures, and core utilities

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]  # Conversation history
    critic_score: int                                    # Evaluation score from Critic
    loop_cnt: Annotated[int, operator.add]                # Iteration counter to prevent infinite loops
    score_history: Annotated[List[int], operator.add]    # Tracked scores across feedback loops

class QualificationScore(BaseModel):
    critic_score: int = Field(ge=0, le=10, description="Research score between 0 and 10.")
    feedback: str = Field(description="Detailed instructions for researcher if data is invalid.") # Feedback sub-query for researcher

class ReportMetricScore(BaseModel):
    report_content: str = Field(description="Complete research report in Markdown format.")
    reliability_index: int = Field(ge=0, le=10, description="Calculated Reliability Index (0-10).")
    iterative_improvement_index: int = Field(ge=0, le=10, description="Calculated Iterative Improvement Index (0-10).")

# Search tool instance
tavily_tool = TavilySearch(max_results=4, output_format="result")

def save_markdown_report(report_content: str, topic: str, topic_index: int, model_type: str = "MAS"):
    """
    Saves report content as a .md file inside a model-specific directory.
    """
    if model_type.upper() == "MAS":
        output_dir = "mas_reports"
        prefix = "MAS"
        title_header = "# [Multi-Agent System Research Report]\n\n"
    else:
        output_dir = "baseline_reports"
        prefix = "Baseline"
        title_header = "# [Baseline Single-Pass Research Report]\n\n"
        
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    clean_topic = re.sub(r'[\\/*?:"<>|]', "", topic).replace(" ", "_")[:30]
    filename = f"Topic_{topic_index:02d}_{prefix}_{clean_topic}.md"
    filepath = os.path.join(output_dir, filename)
    
    final_content = title_header + report_content
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(final_content)
        
    print(f"[{prefix}] Report saved to: {filepath}")
    return filepath

# Initialize LLM
agent = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0,
    api_key=GOOGLE_API_KEY
)

critic_Agent = agent.with_structured_output(QualificationScore)
writer_Agent = agent.with_structured_output(ReportMetricScore)

---

## Step 3. Agent Node & Workflow Implementations

In [ ]:
# Agent Nodes: Researcher

def extract_clean_text(content) -> str:
    """
    Extract pure text content from structured Gemini response objects.
    """
    if isinstance(content, str):
        return content
    elif isinstance(content, list):
        text_parts = []
        for part in content:
            if isinstance(part, dict) and 'text' in part:
                text_parts.append(part['text'])
            elif hasattr(part, 'text'):
                text_parts.append(part.text)
            elif isinstance(part, str):
                text_parts.append(part)
        return "".join(text_parts)
    elif hasattr(content, 'text'):
        return content.text
    return str(content)


def researcher_node(state: AgentState):
    messages = state["messages"]
    user_query = messages[0].content
    latest_message = messages[-1].content if messages else ""

    past_research = next(
        (extract_clean_text(m.content) for m in reversed(messages) 
         if isinstance(m, AIMessage) and "[CRITIC_FEEDBACK]" not in extract_clean_text(m.content)), 
        "None"
    )

    if "[CRITIC_FEEDBACK]" in latest_message:
        query_gen_prompt = f"""
        Original Query: {user_query}
        Critic Feedback: {latest_message}
        Based on the feedback and past research, generate a concise search query to find ONLY the missing data.
        Output ONLY the raw search query text without any formatting.
        """
        raw_query_output = agent.invoke([
            SystemMessage(content="You are a precise web search query generator."),
            HumanMessage(content=query_gen_prompt)
        ]).content
        
        search_query = extract_clean_text(raw_query_output)
        search_query = search_query.replace('"', '').replace("'", "").replace("```", "").strip()
    else:
        search_query = user_query

    print(f"--- Senior researcher searching for: {search_query} ---")

    try:
        new_raw_data = str(tavily_tool.invoke(search_query)) or "No results returned."
    except Exception as e:
        print(f"Tavily Search Error: {e}")
        new_raw_data = "No new data found due to search tool error."

    merge_prompt = f"""You are a Fact-Driven Research Synthesizer.
    Upgrade past research by injecting verified URLs and facts from New Search Data.

    [DATA COMPENDIUM]
    - Initial User Query: {user_query}
    - Recent Critic Feedback: {latest_message if "[CRITIC_FEEDBACK]" in latest_message else "None"}
    - Past Research Framework: {past_research}
    - New Search Data: {new_raw_data}

    Synthesize the final comprehensive research report.
    """

    raw_organized_content = agent.invoke([
        SystemMessage(content="You are a rigorous academic researcher."),
        HumanMessage(content=merge_prompt)
    ]).content

    final_clean_text = extract_clean_text(raw_organized_content)

    return {
        "messages": [AIMessage(content=final_clean_text)],
        "loop_cnt": 1
    }

In [ ]:
# Agent Nodes: Critic

def critic_node(state: AgentState):
    """
    Evaluates gathered research data based on strict quality criteria.
    """
    messages = state["messages"]
    user_query = messages[0].content

    last_research_content = ""
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and "[CRITIC_FEEDBACK]" not in msg.content:
            last_research_content = msg.content
            break

    if not last_research_content and messages:
        last_research_content = messages[-1].content

    critic_prompt = f"""You are a strict and conservative Quality Auditor for Research Data.
    Your goal is to ensure world-class academic rigor. Do NOT give high scores easily.

    Evaluate the gathered research data based on these STRICT criteria:
    1. Accuracy & Grounding (Weight: 40%): Every single major claim must have an explicit inline markdown URL link. If facts are presented without direct source URLs, automatically deduct at least 3 points.
    2. Temporal Accuracy (Weight: 30%): Check for any future date hallucinations or speculative timeline errors (e.g., confusing future estimates with historical facts).
    3. Data Sufficiency (Weight: 30%): Does it systematically address ALL sub-requirements in '{user_query}'? If any specific metric, table, or timeline is missing or vague, the score MUST NOT exceed 6.

    [Scoring Guideline]
    - Score 0-5: Missing core requirements, poor grounding, or timeline hallucinations.
    - Score 6-7: Good general summary, but lacks specific quantitative data, deep comparison, or verified source URLs.
    - Score 8-10: Publication-ready. Perfectly grounded with multiple official URLs, no timeline errors, and complete coverage.

    Initial User Query: {user_query}
    Research Data to Evaluate: {last_research_content}

    Provide an integer score between 0 and 10. If score < 8, provide concrete, actionable feedback instructing the researcher which exact URLs or missing data to search next.
    """
    
    print("--- Critic node executing ---")
    result = critic_Agent.invoke([HumanMessage(content=critic_prompt)])

    feedback_message = []
    if result.critic_score < 8:
        feedback_message = [HumanMessage(content=f"[CRITIC_FEEDBACK] Score: {result.critic_score}, Feedback: {result.feedback}")]

    print(f"Critic Score Received: {result.critic_score}")
    print("Feedback: ", feedback_message[0].content if feedback_message else f"Passed with score {result.critic_score}")

    return {
        "messages": feedback_message,
        "critic_score": result.critic_score,
        "score_history": [result.critic_score]
    }

In [ ]:
# Agent Nodes: Technical Writer

def writer_node(state: AgentState):
    """
    Technical Writer node using Structured Output.
    Calculates custom quality metrics when operating in MAS mode.
    """
    if not isinstance(state, dict):
        messages = state.messages if hasattr(state, 'messages') else []
        scores = []
    else:
        messages = state.get("messages", [])
        scores = state.get("score_history", [])

    last_research_content = ""
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and "[CRITIC_FEEDBACK]" not in msg.content:
            last_research_content = msg.content
            break
            
    if not last_research_content and messages:
        last_research_content = messages[-1].content

    is_multi_agent = len(scores) > 0

    print("--- Technical writer executing ---")

    if is_multi_agent:
        score_trend = f"Score History: {scores}"
        
        writer_prompt = f"""You are a Technical Writer and Data Quality Analyst.
        1. Synthesize the research report in Markdown based on the research content.
        2. Calculate 'Reliability Index Metric' and 'Iterative Improvement Index Metric' (integer, 0-10) based on the Score History: {score_trend}.

        [Research Content]
        {last_research_content}
        """

        result: ReportMetricScore = writer_Agent.invoke([HumanMessage(content=writer_prompt)])

        rel_score = result.reliability_index
        imp_score = result.iterative_improvement_index
        avg_writer_score = round((rel_score + imp_score) / 2)

        print(f"[MAS Mode] Reliability={rel_score}, Improvement={imp_score} -> Average={avg_writer_score}")

        full_report_content = (
            f"{result.report_content}\n\n"
            f"# Custom Quality Metrics Evaluation\n"
            f"- **Reliability Index Metric**: {rel_score} / 10\n"
            f"- **Iterative Improvement Index Metric**: {imp_score} / 10\n"
            f"- **Average Quality Metric**: {avg_writer_score} / 10"
        )

        return {
            "messages": [AIMessage(content=full_report_content)],
            "critic_score": avg_writer_score,
            "score_history": [avg_writer_score]
        }
    else:
        print("[Single Agent Mode] Skipping Custom Quality Metrics calculation.")
        
        writer_prompt = f"""You are a Technical Writer.
        Synthesize a well-organized research report in Markdown format.

        [Research Content]
        {last_research_content}
        """
        
        single_response = agent.invoke([HumanMessage(content=writer_prompt)])
        single_content = single_response.content

        return {
            "messages": [AIMessage(content=str(single_content))]
        }

In [ ]:
# Define & Compile Multi-Agent Workflow

def validationProcess(state: AgentState):
    if state["critic_score"] >= 8 or state.get("loop_cnt", 0) >= 5:
        return "OKAY"
    else:
        return "AGAIN"

workflow = StateGraph(AgentState)

workflow.add_node("researcher", researcher_node)
workflow.add_node("critic", critic_node)
workflow.add_node("writer", writer_node)

workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "critic")
workflow.add_conditional_edges(
    "critic",
    validationProcess,
    {
        "AGAIN": "researcher",
        "OKAY": "writer"
    }
)
workflow.add_edge("writer", END)

app = workflow.compile()
print("Multi-Agent LangGraph Workflow compiled successfully.")

In [ ]:
# Define & Compile Baseline Workflow (No Critic, No Feedback Loop)
from langgraph.graph import StateGraph, END

baseline_workflow = StateGraph(AgentState)

baseline_workflow.add_node("researcher", researcher_node)
baseline_workflow.add_node("writer", writer_node)

baseline_workflow.set_entry_point("researcher")
baseline_workflow.add_edge("researcher", "writer")
baseline_workflow.add_edge("writer", END)

baseline_app = baseline_workflow.compile()
print("Baseline (Single-Pass) LangGraph Workflow compiled successfully.")

---

## Step 4. Run Web Research Workflows 

In [ ]:
# Run Baseline Benchmark Across Topics

single_score_histories = {}

if os.path.exists("research-topics.txt"):
    with open("research-topics.txt", "r", encoding="utf-8") as f:
        topics = [line.strip() for line in f if line.strip()]

print(f"Starting Baseline Workflow Benchmark for {len(topics)} topics...\n")

for i, topic in enumerate(topics, 1):
    print(f"\n{'='*70}")
    print(f"[Baseline - Topic {i}/{len(topics)}] {topic[:60]}...")
    print(f"{'='*70}")
    
    baseline_result = baseline_app.invoke(
        {"messages": [HumanMessage(content=topic)]},
        config={"recursion_limit": 10}
    )
    
    baseline_report = baseline_result['messages'][-1].content
    
    saved_path = save_markdown_report(
        report_content=baseline_report, 
        topic=topic, 
        topic_index=i, 
        model_type="Baseline"
    )
    
    eval_prompt = f"""You are an impartial Quality Auditor.
    Evaluate the following research report for the query: '{topic}'.
    Provide an integer score between 0 and 10 based on accuracy, grounding, and completeness.
    
    [Report to Evaluate]
    {baseline_report}
    """
    eval_result = critic_Agent.invoke([HumanMessage(content=eval_prompt)])
    baseline_score = eval_result.critic_score
    
    print(f"Baseline Final Score: {baseline_score} (No Loop)")
    
    single_score_histories[f"Topic {i}"] = [baseline_score]

print(f"\n{'='*70}")
print("Baseline Workflow Benchmark Completed!")
print(f"{'='*70}\n")

In [ ]:
# Run Multi-Agent System Benchmark Across Topics
topics = []
if os.path.exists("research-topics.txt"):
    with open("research-topics.txt", "r", encoding="utf-8") as f:
        topics = [line.strip() for line in f if line.strip()]

all_score_histories = {}

print(f"Total {len(topics)} research topics loaded.\n")

for i, topic in enumerate(topics, 1):
    print(f"\n{'='*70}")
    print(f"[Topic {i}/{len(topics)}] Researching: {topic[:60]}...")
    print(f"{'='*70}")
    
    result = app.invoke(
        {"messages": [HumanMessage(content=topic)]},
        config={"recursion_limit": 25}
    )
    
    final_report = result['messages'][-1].content
    score_history = result.get('score_history', [])
    
    saved_path = save_markdown_report(
        report_content=final_report, 
        topic=topic, 
        topic_index=i, 
        model_type="MAS"
    )
    
    topic_label = f"Topic {i}"
    all_score_histories[topic_label] = score_history

print(f"\n{'='*70}")
print("All Research Tasks Completed! Generating Comparative Metric Report...")
print(f"{'='*70}\n")

---

## Step 5. Visualization & Analysis

In [ ]:
# Comparative Visualization: Mean Score Comparison

def plot_mean_score_comparison(all_score_histories: dict, single_score_histories: dict):
    """
    Plot mean score comparison between Baseline and MAS.
    """
    mas_final_scores = [history[-1] for history in all_score_histories.values() if history]
    baseline_final_scores = [history[-1] for history in single_score_histories.values() if history]

    if not mas_final_scores or not baseline_final_scores:
        print("Warning: Score histories incomplete. Cannot compute comparison.")
        return

    mean_baseline = np.mean(baseline_final_scores)
    mean_mas = np.mean(mas_final_scores)
    
    improvement_pct = ((mean_mas - mean_baseline) / mean_baseline) * 100 if mean_baseline > 0 else 0

    plt.style.use('seaborn-v0_8-muted')
    fig, ax = plt.subplots(figsize=(7, 6))

    categories = ['Baseline\n(Single-Pass)', 'Multi-Agent System\n(Feedback Loop-Based)']
    means = [mean_baseline, mean_mas]
    colors = ['#A9CCE3', '#2E86C1']

    bars = ax.bar(categories, means, color=colors, width=0.45, edgecolor='black', linewidth=1.2)

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.2,
                f'{height:.2f} pts',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_ylabel('Mean Critic Score (0 - 10)', fontsize=12)
    ax.set_title(f'Overall Reliability Improvement: +{improvement_pct:.1f}%', fontsize=14, pad=15, fontweight='bold', color='#1B4F72')
    ax.set_ylim(0, 11)
    ax.axhline(y=8, color='red', linestyle='--', alpha=0.5, label='Pass Threshold (Score 8)')
    ax.grid(axis='y', linestyle=':', alpha=0.7)
    ax.legend(loc='upper left')

    plt.tight_layout()
    plt.show()

    print("[Performance Summary]")
    print(f"- Baseline Mean Score: {mean_baseline:.2f} / 10")
    print(f"- MAS Mean Score:      {mean_mas:.2f} / 10")
    print(f"- Reliability Growth:  +{improvement_pct:.1f}% Improvement\n")

plot_mean_score_comparison(all_score_histories, single_score_histories)

In [ ]:
# Interpolated Time-Normalization Performance Trajectory Plot

def plot_mas_reliability_metrics(all_score_histories: dict, num_points: int = 5):
    """
    Plot time-normalized mean trajectory curve across all research topics.
    """
    if not all_score_histories:
        print("Warning: No MAS score histories available.")
        return

    common_x = np.linspace(0.0, 1.0, num_points)
    interpolated_matrix = []

    for topic, history in all_score_histories.items():
        if not history:
            continue
            
        n = len(history)
        if n == 1:
            interpolated_scores = np.full(num_points, history[0])
        else:
            orig_x = np.linspace(0.0, 1.0, n)
            f = interpolate.interp1d(orig_x, history, kind='linear')
            interpolated_scores = f(common_x)
            
        interpolated_matrix.append(interpolated_scores)

    if not interpolated_matrix:
        print("Warning: Could not process score histories.")
        return

    mean_trajectory = np.mean(interpolated_matrix, axis=0)
    progress_labels = [f"{int(p * 100)}%" for p in common_x]

    plt.style.use('seaborn-v0_8-muted')
    fig, ax = plt.subplots(figsize=(9, 5.5))

    ax.plot(progress_labels, mean_trajectory, marker='o', markersize=8, linewidth=3, 
            color='#2E86C1', label='Normalized Mean Trajectory')

    for i, mean_val in enumerate(mean_trajectory):
        ax.text(i, mean_val + 0.3, f"{mean_val:.2f} pts", ha='center', va='bottom', 
                fontsize=10, fontweight='bold', color='#1B4F72')

    initial_val = mean_trajectory[0]
    final_val = mean_trajectory[-1]
    total_delta = final_val - initial_val
    improvement_pct = (total_delta / initial_val) * 100 if initial_val > 0 else 0

    ax.set_title(f'MAS Self-Improvement Trajectory (Normalized)\nOverall Growth: +{improvement_pct:.1f}% (+{total_delta:.2f} pts)', 
                 fontsize=13, fontweight='bold', pad=15, color='#1B4F72')
    ax.set_xlabel('Research & Audit Progress Rate (%)', fontsize=11, labelpad=10)
    ax.set_ylabel('Mean Critic Score (0 - 10)', fontsize=11)
    ax.set_ylim(0, 11)
    ax.axhline(y=8, color='#E74C3C', linestyle='--', alpha=0.7, linewidth=1.5, label='Pass Threshold (Score 8.0)')
    
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='none')

    plt.tight_layout()
    plt.show()

    print("[Normalized Process Performance Summary]")
    print(f"- Initial Trajectory Mean (0%):   {initial_val:.2f} / 10")
    print(f"- Final Trajectory Mean (100%):  {final_val:.2f} / 10")
    print(f"- Total Score Improvement:      +{total_delta:.2f} pts (+{improvement_pct:.1f}%)\n")

plot_mas_reliability_metrics(all_score_histories)